# Preparación del Dataset Facial Multi-Vista

Este notebook permite:

- Verificar que la estructura de `data/raw` sea correcta antes de procesar.
- Ejecutar la extracción de fotogramas desde los videos.
- Ejecutar el recorte facial fotograma a fotograma.
- Revisar conteos de videos, frames y rostros procesados por persona y vista.
- Visualizar muestras del dataset local solo para revisión de calidad.

Ajusta los parámetros en la siguiente celda antes de ejecutar cualquier paso del pipeline.

In [ ]:
# --- Parámetros de ejecución ---
# Cambia estos valores antes de ejecutar el pipeline.
# Las tareas pesadas están en False por seguridad; actívalas cuando tengas los videos listos.

RUN_VALIDATE_STRUCTURE = True    # Verificar carpetas y nombres (rápido, seguro ejecutar)
RUN_EXTRACT_FRAMES     = False   # Extraer frames desde videos (lento, requiere videos)
RUN_FACE_CROP          = False   # Detectar y recortar rostros (lento, requiere frames)

# Extracción de frames
FRAME_INTERVAL   = 10     # Tomar 1 frame cada N frames del video
OVERWRITE_FRAMES = False  # Si True, sobreescribe frames ya existentes

# Recorte facial
FACE_MARGIN     = 0.20   # Margen adicional alrededor del rostro detectado (proporción)
OVERWRITE_FACES = False  # Si True, sobreescribe rostros ya recortados

# Visualización de muestras
SHOW_FACE_SAMPLES = False  # Mostrar muestra de rostros locales (desactivado por privacidad)

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.config import (
    IMAGE_SIZE,
    PAIRS_DIR,
    PROCESSED_DATASET_DIR,
    PROJECT_ROOT,
    RAW_DATASET_DIR,
    SUPPORT_SET_DIR,
    SUPPORTED_FACE_VIEWS,
)

# Extensiones de archivo reconocidas
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

# Inicializar listas de conteo (se actualizan en celdas posteriores)
raw_records    = []
frames_records = []
faces_records  = []

print(f'PROJECT_ROOT          : {PROJECT_ROOT}')
print(f'RAW_DATASET_DIR       : {RAW_DATASET_DIR}')
print(f'PROCESSED_DATASET_DIR : {PROCESSED_DATASET_DIR}')
print(f'SUPPORTED_FACE_VIEWS  : {SUPPORTED_FACE_VIEWS}')
print(f'IMAGE_SIZE            : {IMAGE_SIZE}')

In [ ]:
def run_command(command):
    """Ejecuta un comando del pipeline y muestra la salida completa al terminar."""
    cmd_str = ' '.join(command)
    print(f'$ {cmd_str}')
    print('-' * 60)
    result = subprocess.run(
        command,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout or '(sin salida)')
    print('-' * 60)
    if result.returncode != 0:
        print(f'[ERROR] Código de salida: {result.returncode}')
    else:
        print('[OK] Proceso completado sin errores.')


def count_raw_videos(raw_dir):
    """Recorre data/raw y devuelve conteos de videos por persona y vista."""
    records = []
    if not raw_dir.exists():
        return records
    for person_dir in sorted(raw_dir.iterdir()):
        if not person_dir.is_dir():
            continue
        for view_dir in sorted(person_dir.iterdir()):
            if not view_dir.is_dir():
                continue
            n = sum(
                1 for f in view_dir.iterdir()
                if f.is_file() and f.suffix.lower() in VIDEO_EXTENSIONS
            )
            records.append({'person_id': person_dir.name, 'view': view_dir.name, 'videos': n})
    return records


def count_processed(processed_dir, subfolder):
    """Cuenta archivos en la subcarpeta indicada (frames o faces) por persona y vista."""
    records = []
    if not processed_dir.exists():
        return records
    for person_dir in sorted(processed_dir.iterdir()):
        if not person_dir.is_dir():
            continue
        for view_dir in sorted(person_dir.iterdir()):
            if not view_dir.is_dir():
                continue
            target = view_dir / subfolder
            n = sum(1 for f in target.iterdir() if f.is_file()) if target.exists() else 0
            records.append({'person_id': person_dir.name, 'view': view_dir.name, subfolder: n})
    return records

## Estructura esperada del dataset

Los videos deben colocarse en `data/raw/` siguiendo esta jerarquía exacta.
El dataset es **privado y local**: no debe subirse a GitHub ni incluirse en ningún commit.

```
data/
└── raw/
    ├── user_001/
    │   ├── frontal/  video_01.mp4
    │   ├── left/     video_01.mp4
    │   ├── right/    video_01.mp4
    │   └── mixed/    video_01.mp4
    └── user_002/
        ├── frontal/  video_01.mp4
        ├── left/     video_01.mp4
        ├── right/    video_01.mp4
        └── mixed/    video_01.mp4
```

**Parámetros de grabación recomendados:**

| Parámetro | Valor recomendado |
|-----------|-------------------|
| Personas | 10 |
| Vistas por persona | 4 (frontal, left, right, mixed) |
| Duración por video | ~12 segundos |
| Resolución | 1080p |
| FPS | 30 |

El `person_id` puede ser cualquier identificador sin espacios ni caracteres especiales (p. ej. `user_001`).

In [ ]:
# Verificar existencia de las carpetas principales del dataset
dirs_to_check = {
    'data/raw':         RAW_DATASET_DIR,
    'data/processed':   PROCESSED_DATASET_DIR,
    'data/pairs':       PAIRS_DIR,
    'data/support_set': SUPPORT_SET_DIR,
}

print('=== Estado de carpetas del dataset ===')
for label, path in dirs_to_check.items():
    if path.exists():
        n_items = sum(1 for _ in path.iterdir())
        status = f'OK  ({n_items} elementos)'
    else:
        status = 'FALTA — no existe todavía'
    print(f'  {label:<22} {status}')

In [ ]:
# Contar videos disponibles en data/raw, agrupados por persona y vista
raw_records = count_raw_videos(RAW_DATASET_DIR)

if raw_records:
    df_raw = pd.DataFrame(raw_records)
    present_views = [v for v in SUPPORTED_FACE_VIEWS if v in df_raw['view'].unique()]
    df_pivot = (
        df_raw
        .pivot_table(index='person_id', columns='view', values='videos', aggfunc='sum', fill_value=0)
        .reindex(columns=present_views, fill_value=0)
    )
    df_pivot.columns.name = None
    df_pivot['total_videos'] = df_pivot.sum(axis=1)
    n_people = len(df_pivot)
    n_videos = int(df_pivot['total_videos'].sum())
    print(f'Personas encontradas : {n_people}')
    print(f'Videos totales       : {n_videos}')
    print()
    print(df_pivot.to_string())
else:
    print('No se encontraron videos en data/raw/.')
    print('Coloca los archivos de video antes de continuar con el pipeline.')

In [ ]:
# Paso 1 del pipeline: validar la estructura del dataset
cmd_validate = ['python', '-m', 'src.dataset.validate_structure']

if RUN_VALIDATE_STRUCTURE:
    run_command(cmd_validate)
else:
    cmd_str = ' '.join(cmd_validate)
    print('RUN_VALIDATE_STRUCTURE = False. Comando a ejecutar cuando esté listo:')
    print(f'  $ {cmd_str}')

In [ ]:
# Paso 2 del pipeline: extraer fotogramas desde los videos
cmd_frames = ['python', '-m', 'src.preprocessing.extract_frames', '--interval', str(FRAME_INTERVAL)]
if OVERWRITE_FRAMES:
    cmd_frames.append('--overwrite')

if RUN_EXTRACT_FRAMES:
    run_command(cmd_frames)
else:
    cmd_str = ' '.join(cmd_frames)
    print('RUN_EXTRACT_FRAMES = False. Actívalo en la celda de parámetros para ejecutar.')
    print(f'  $ {cmd_str}')

In [ ]:
# Contar fotogramas extraídos en data/processed/<person_id>/<view>/frames/
frames_records = count_processed(PROCESSED_DATASET_DIR, 'frames')

if frames_records:
    df_frames = pd.DataFrame(frames_records)
    total_frames = df_frames['frames'].sum()
    print(f'Fotogramas extraídos: {total_frames}')
    print()
    print(df_frames.to_string(index=False))
else:
    print('No hay fotogramas en data/processed/.')
    print('Activa RUN_EXTRACT_FRAMES = True y vuelve a ejecutar.')

In [ ]:
# Paso 3 del pipeline: detectar y recortar rostros fotograma a fotograma
cmd_crop = ['python', '-m', 'src.preprocessing.face_crop', '--margin', str(FACE_MARGIN)]
if OVERWRITE_FACES:
    cmd_crop.append('--overwrite')

if RUN_FACE_CROP:
    run_command(cmd_crop)
else:
    cmd_str = ' '.join(cmd_crop)
    print('RUN_FACE_CROP = False. Actívalo en la celda de parámetros para ejecutar.')
    print(f'  $ {cmd_str}')

In [ ]:
# Contar rostros recortados en data/processed/<person_id>/<view>/faces/
faces_records = count_processed(PROCESSED_DATASET_DIR, 'faces')

if faces_records:
    df_faces = pd.DataFrame(faces_records)
    total_faces = df_faces['faces'].sum()
    print(f'Rostros recortados: {total_faces}')
    print()
    print(df_faces.to_string(index=False))
else:
    print('No hay rostros recortados en data/processed/.')
    print('Activa RUN_FACE_CROP = True y vuelve a ejecutar.')

In [ ]:
# Tabla resumen combinada: videos, frames y rostros por persona y vista
def build_summary(raw_dir, processed_dir):
    r  = count_raw_videos(raw_dir)
    f  = count_processed(processed_dir, 'frames')
    fa = count_processed(processed_dir, 'faces')

    if not r and not f and not fa:
        return pd.DataFrame(columns=['person_id', 'view', 'videos', 'frames', 'faces'])

    df_r  = pd.DataFrame(r)  if r  else pd.DataFrame(columns=['person_id', 'view', 'videos'])
    df_f  = pd.DataFrame(f)  if f  else pd.DataFrame(columns=['person_id', 'view', 'frames'])
    df_fa = pd.DataFrame(fa) if fa else pd.DataFrame(columns=['person_id', 'view', 'faces'])

    # Reunir todos los pares únicos (person_id, view) de las tres fuentes
    keys = (
        pd.concat([
            df_r[['person_id', 'view']],
            df_f[['person_id', 'view']],
            df_fa[['person_id', 'view']],
        ])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    on = ['person_id', 'view']
    df = (
        keys
        .merge(df_r,  on=on, how='left')
        .merge(df_f,  on=on, how='left')
        .merge(df_fa, on=on, how='left')
    )
    df[['videos', 'frames', 'faces']] = df[['videos', 'frames', 'faces']].fillna(0).astype(int)
    return df.sort_values(on).reset_index(drop=True)


df_summary = build_summary(RAW_DATASET_DIR, PROCESSED_DATASET_DIR)

if df_summary.empty:
    print('No hay datos disponibles todavía.')
    print('Coloca los videos y ejecuta el pipeline completo para ver el resumen.')
else:
    print('=== Resumen del dataset ===')
    print(df_summary.to_string(index=False))
    print()
    totals = df_summary[['videos', 'frames', 'faces']].sum()
    v, fr, fa = int(totals['videos']), int(totals['frames']), int(totals['faces'])
    print(f'Totales -> videos: {v}  frames: {fr}  faces: {fa}')

In [ ]:
# Gráficos de resumen: videos, frames y rostros por vista
def plot_by_view(ax, records, col, title):
    """Dibuja un gráfico de barras por vista o muestra mensaje si no hay datos."""
    if records:
        df = pd.DataFrame(records)
        values = df.groupby('view')[col].sum().reindex(SUPPORTED_FACE_VIEWS, fill_value=0)
        ax.bar(values.index, values.values)
        ax.set_ylim(bottom=0)
    else:
        ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title)
    ax.set_xlabel('Vista')
    ax.set_ylabel('Cantidad')
    ax.tick_params(axis='x', rotation=15)


fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_by_view(axes[0], raw_records,    'videos', 'Videos por vista')
plot_by_view(axes[1], frames_records, 'frames', 'Frames por vista')
plot_by_view(axes[2], faces_records,  'faces',  'Rostros recortados por vista')
plt.suptitle('Estado del dataset por vista', fontsize=12)
plt.tight_layout()
plt.show()

## Revisión visual de rostros

> **Advertencia:** Esta sección muestra rostros reales del dataset local.
> No guardar ni subir el notebook con outputs visibles en esta sección.
> Ejecuta **Kernel → Restart Kernel and Clear All Outputs** antes de hacer `git commit`.

In [ ]:
# Muestra de rostros recortados — solo se ejecuta si SHOW_FACE_SAMPLES = True
if SHOW_FACE_SAMPLES:
    import cv2

    MAX_SAMPLES = 8
    sample_paths = []

    if PROCESSED_DATASET_DIR.exists():
        for p in sorted(PROCESSED_DATASET_DIR.rglob('faces/*')):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                sample_paths.append(p)
                if len(sample_paths) >= MAX_SAMPLES:
                    break

    if sample_paths:
        n_cols = 4
        n_rows = (len(sample_paths) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
        # Normalizar ejes a lista plana independientemente de la forma
        axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

        for ax, img_path in zip(axes_flat, sample_paths):
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is not None:
                ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
            ax.axis('off')
            # Mostrar persona y vista en el título de cada imagen
            parts = img_path.parts
            label = f'{parts[-4]}/{parts[-3]}' if len(parts) >= 4 else img_path.stem
            ax.set_title(label, fontsize=7)

        for ax in axes_flat[len(sample_paths):]:
            ax.axis('off')

        fig.suptitle('Muestra de rostros recortados', fontsize=11)
        plt.tight_layout()
        plt.show()
    else:
        print('No se encontraron imágenes de rostros en data/processed/.')
        print('Ejecuta el recorte facial primero (RUN_FACE_CROP = True).')
else:
    print('Visualización desactivada (SHOW_FACE_SAMPLES = False).')
    print('Activa el parámetro para revisar muestras de forma local.')

## Checklist antes de generar pares

Verifica que se hayan completado todos los pasos antes de continuar:

- [ ] Videos colocados en `data/raw/<person_id>/<view>/`
- [ ] Estructura validada con `validate_structure` sin errores
- [ ] Frames extraídos en `data/processed/<person_id>/<view>/frames/`
- [ ] Rostros recortados en `data/processed/<person_id>/<view>/faces/`
- [ ] Revisión visual de calidad realizada (muestras sin errores de detección)
- [ ] Outputs del notebook limpiados antes del commit

**Siguiente paso:** generar pares de entrenamiento.

```
python -m src.dataset.build_pairs --overwrite
```